In [3]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import nltk

## 1. Loading the dataset

In [4]:
#Loading the dataset
file_path = "Datasets/amazon_review_small.txt"
df = pd.read_csv(file_path, sep=",", quotechar='"', engine="python", header=None, names=["rating","title", "review"])



In [4]:
print(df.head(2))
print(f"\nLoaded {len(df)} rows and {df.shape[1]} columns from {file_path}")

   rating                    title  \
0       1          mens ultrasheer   
1       4  Surprisingly delightful   

                                              review  
0  This model may be ok for sedentary types, but ...  
1  This is a fast read filled with unexpected hum...  

Loaded 650000 rows and 3 columns from Datasets/amazon_review_small.txt


## 2. Exploring Data

In [5]:
df["rating"].value_counts()

rating
1    130000
4    130000
2    130000
3    130000
5    130000
Name: count, dtype: int64

As we cam see we have rating 1-5 and each rating has equal number of reviews. 

In [6]:
df.isnull().sum()

rating     0
title     26
review     0
dtype: int64

In [8]:
#Fill missing values with empty string
df["title"] = df["title"].fillna("")

## 3. Cleaning Data

In [9]:
#Creating a copy of dataset and Combining title and review into a single text column
df_work = df.copy()
df_work["text"] = df_work["title"] + " " + df_work["review"]
#Selecting only the text and rating columns for further processing
df_work = df_work[["text", "rating"]]
df_work["text"][0]

"mens ultrasheer This model may be ok for sedentary types, but I'm active and get around alot in my job - consistently found these stockings rolled up down by my ankles! Not Good!! Solution: go with the standard compression stocking, 20-30, stock #114622. Excellent support, stays up and gives me what I need. Both pair of these also tore as I struggled to pull them up all the time. Good riddance/bad investment!"

In [10]:
#Function to clean the text data
import re
def clean_text(text):
    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)
    # Remove non-alphanumeric characters with spaces to preserve proper word separation
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    # Convert to lowercase
    text = text.lower()
    #Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [11]:
#Apply cleaning function to the text column
df_work["text"] = df_work["text"].apply(clean_text)

## 4. Splitting Data

In [12]:
# Separate features (X) and target (y)
X = df_work["text"]
y = df_work["rating"]

# First split: 80% train, 20% test
from sklearn.model_selection import train_test_split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Second split (on training data): 80% train, 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, random_state=42
)

print("Train size:", X_train.shape[0])
print("y_train shape:", y_train.shape)
print("Validation size:", X_val.shape[0])
print("y_val shape:", y_val.shape)
print("Test size:", X_test.shape[0])

Train size: 416000
y_train shape: (416000,)
Validation size: 104000
y_val shape: (104000,)
Test size: 130000


## 5. Vectorization
### Strategy

Initially, we will use TF-IDF for the **base model** and check accuracy. Later, if needed, we will switch to **Word2Vec** or other advanced embedding techniques.
### What is TF-IDF?

**TF-IDF (Term Frequency – Inverse Document Frequency)** converts raw text into numeric vectors that machine learning models can process.

It scores each word in a document using two components:

- **TF (Term Frequency):** How often a word appears in a single document (review).  
  $TF(t, d) = \dfrac{\text{count of } t \text{ in } d}{\text{total words in } d}$

- **IDF (Inverse Document Frequency):** Penalizes words that appear in many documents (e.g. "the", "is"), giving higher weight to rare, meaningful words.  
  $IDF(t) = \log\left(\dfrac{1 + N}{1 + df(t)}\right) + 1$

- **TF-IDF score:**  
  $\text{TF-IDF}(t, d) = TF(t, d) \times IDF(t)$

### How `TfidfVectorizer` works

1. Scans all training reviews and builds a **vocabulary** of unique words.
2. Ranks words by frequency and keeps the top **`max_features`** words.
3. Represents each review as a vector of TF-IDF scores — one value per vocabulary word.
4. Words not seen during training are ignored at transform time.

### Settings used here

| Parameter | Value | Meaning |
|---|---|---|
| `max_features` | 5000 | Keep only top 5000 most frequent words as vocab |
| `analyzer` | word *(default)* | Tokenize by words, not characters |
| `token_pattern` | `\b[a-zA-Z0-9]{2,}\b` *(default)* | Words with 2+ alphanumeric chars |


In [19]:
# Vectorization using TF-IDF (fit on train, transform train/validation/test)
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)  # Limit to top 5000 features
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_val_tfidf shape:", X_val_tfidf.shape)
print("X_test_tfidf shape:", X_test_tfidf.shape)

X_train_tfidf shape: (416000, 5000)
X_val_tfidf shape: (104000, 5000)
X_test_tfidf shape: (130000, 5000)


In [20]:
# Displaying the word to index mapping for the TF-IDF vectorizer
word_to_index = vectorizer.vocabulary_
print("Sample word to index mapping:", list(word_to_index.items())[:10])


Sample word to index mapping: [('this', np.int64(4455)), ('is', np.int64(2326)), ('not', np.int64(2983)), ('book', np.int64(540)), ('about', np.int64(81)), ('being', np.int64(457)), ('fat', np.int64(1648)), ('its', np.int64(2337)), ('yourself', np.int64(4988)), ('am', np.int64(214))]


## 6. Using a base model to train data.

In [21]:
#Training a simple logistic regression model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000, n_jobs=-1)
model.fit(X_train_tfidf, y_train)

d:\AmazonReviewRatingPredictions\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [52]:
from sklearn.metrics import classification_report

def evaluate_model(model, X, y , label: str = "Dataset"):
    """
    Generic model evaluator.

    Parameters:
        model : fitted sklearn model
        X     : input feature vectors (TF-IDF sparse matrix or any array-like)
        y     : true labels
        label : name to display in the output (e.g. "Validation", "Test")

    Returns:
        report : classification report as a dictionary
    """
    preds = model.predict(X)
    report =  classification_report(y, preds)
    print("Model name: {}, {} Classification Report:".format(model.__class__.__name__, label))
    print(report)
    return report

# Example usage:
val_report = evaluate_model(model, X_val_tfidf, y_val, label="Validation")
test_report = evaluate_model(model, X_test_tfidf, y_test, label="Test")

Model name:LogisticRegression, Validation Classification Report:
              precision    recall  f1-score   support

           1       0.63      0.67      0.65     20871
           2       0.46      0.44      0.45     20793
           3       0.46      0.44      0.45     20674
           4       0.48      0.45      0.46     20842
           5       0.63      0.70      0.66     20820

    accuracy                           0.54    104000
   macro avg       0.53      0.54      0.53    104000
weighted avg       0.53      0.54      0.54    104000

Model name:LogisticRegression, Test Classification Report:
              precision    recall  f1-score   support

           1       0.63      0.67      0.65     25890
           2       0.46      0.43      0.44     25983
           3       0.46      0.43      0.44     26226
           4       0.47      0.45      0.46     25849
           5       0.63      0.70      0.66     26052

    accuracy                           0.54    130000
   macr

### Base Model Results — Logistic Regression + TF-IDF

The classification report shows per-class **precision**, **recall**, and **F1-score** for all 5 rating classes (1–5).

- **Precision:** Of all reviews predicted as rating X, how many actually were X.
- **Recall:** Of all actual rating X reviews, how many did the model correctly find.
- **F1-score:** Harmonic mean of precision and recall — the main metric to watch.

**Key observations:**
- Extreme ratings (1 and 5) are typically predicted with higher accuracy as they contain stronger sentiment signals.
- Middle ratings (2, 3, 4) tend to be harder to distinguish and usually show lower F1-scores.
- **Macro avg F1** gives an equal-weight view across all classes.
- **Weighted avg F1** accounts for class size and is the best single number to compare models overall.

This serves as the **baseline** — further steps (hyperparameter tuning, better embeddings) should aim to improve on these numbers.

## Iteration 1 — N-gram Feature Expansion

**What changed:** Added **bigrams** (`ngram_range=(1, 2)`) to the TF-IDF vectorizer alongside unigrams.

**Why this helps:**
- Single words (unigrams) miss context — e.g. `"not good"` gets split into `"not"` and `"good"` separately.
- Bigrams capture word pairs like `"not good"`, `"highly recommend"`, `"waste money"`, giving the model richer signal.

**Expected impact:** Moderate accuracy improvement, especially for mid-range ratings (2, 3, 4) that rely more on phrasing than individual words.

In [ ]:
# Iteration 1: TF-IDF with unigrams + bigrams
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer_v1 = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 3)    # unigrams + bigrams
)
X_train_tfidf_v1 = vectorizer_v1.fit_transform(X_train)
X_val_tfidf_v1   = vectorizer_v1.transform(X_val)
X_test_tfidf_v1  = vectorizer_v1.transform(X_test)

model_v1 = LogisticRegression(max_iter=1000, n_jobs=-1)
model_v1.fit(X_train_tfidf_v1, y_train)

val_report_v1  = evaluate_model(model_v1, X_val_tfidf_v1,  y_val,  label="Validation — Iteration 1 (Bigrams)")
test_report_v1 = evaluate_model(model_v1, X_test_tfidf_v1, y_test, label="Test — Iteration 1 (Bigrams)")

### Iteration 1 Summary

Adding bigrams improved the model over the base TF-IDF setup on both validation and test data.

- Validation accuracy increased from **0.5384** to **0.5535**.
- Test accuracy increased from **0.5356** to **0.5538**.
- Validation weighted F1 increased from **0.5351** to **0.5503**.
- Test weighted F1 increased from **0.5320** to **0.5503**.

This suggests that phrase-level features such as two-word combinations are helping the model capture sentiment more effectively than single-word features alone.

## Iteration 2 — Stopword Removal + Document Frequency Tuning

**What changed:** This iteration keeps TF-IDF with unigrams+bigrams and adds English stopword removal (`stop_words='english'`). It also tunes `min_df` and `max_df` to keep terms that are informative for this dataset.

- `min_df`: removes very rare terms that may add noise.
- `max_df`: removes very common terms that carry little signal.

We select the best (`min_df`, `max_df`) pair based on **validation weighted F1** and then report validation and test performance.

In [25]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Candidate values to search
min_df_grid = [1, 2, 3, 5]
max_df_grid = [0.85, 0.90, 0.95, 1.0]

search_rows = []
best_score = -1.0
best_params = None

for min_df in min_df_grid:
    for max_df in max_df_grid:
        # Skip invalid combinations
        if isinstance(max_df, float) and max_df <= 0:
            continue

        vec = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            stop_words="english",
            min_df=min_df,
            max_df=max_df,
        )

        X_train_v2_tmp = vec.fit_transform(X_train)
        X_val_v2_tmp = vec.transform(X_val)

        clf = LogisticRegression(max_iter=1000, n_jobs=-1)
        clf.fit(X_train_v2_tmp, y_train)

        val_pred = clf.predict(X_val_v2_tmp)
        val_metrics = classification_report(y_val, val_pred, output_dict=True)
        val_wf1 = val_metrics["weighted avg"]["f1-score"]
        val_acc = val_metrics["accuracy"]

        search_rows.append(
            {
                "min_df": min_df,
                "max_df": max_df,
                "val_weighted_f1": val_wf1,
                "val_accuracy": val_acc,
                "num_features": X_train_v2_tmp.shape[1],
            }
        )

        if val_wf1 > best_score:
            best_score = val_wf1
            best_params = (min_df, max_df)

# Show grid search results sorted by validation weighted F1
results_v2 = pd.DataFrame(search_rows).sort_values("val_weighted_f1", ascending=False)
display(results_v2.head(10))

best_min_df, best_max_df = best_params
print(f"Best params -> min_df={best_min_df}, max_df={best_max_df}")

# Train final Iteration 2 model with best params
vectorizer_v2 = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=best_min_df,
    max_df=best_max_df,
)

X_train_tfidf_v2 = vectorizer_v2.fit_transform(X_train)
X_val_tfidf_v2 = vectorizer_v2.transform(X_val)
X_test_tfidf_v2 = vectorizer_v2.transform(X_test)

model_v2 = LogisticRegression(max_iter=1000, n_jobs=-1)
model_v2.fit(X_train_tfidf_v2, y_train)

val_report_v2 = evaluate_model(model_v2, X_val_tfidf_v2, y_val, label="Validation — Iteration 2 (Stopwords + DF tune)")
test_report_v2 = evaluate_model(model_v2, X_test_tfidf_v2, y_test, label="Test — Iteration 2 (Stopwords + DF tune)")

d:\AmazonReviewRatingPredictions\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\AmazonReviewRatingPredictions\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\AmazonReviewRatingPredictions\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\AmazonReviewRatingPredictions\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You 

,min_df,max_df,val_weighted_f1,val_accuracy,num_features
0,1,0.85,0.512633,0.51699,5000
1,1,0.90,0.512633,0.51699,5000
2,1,0.95,0.512633,0.51699,5000
3,1,1.00,0.512633,0.51699,5000
4,2,0.85,0.512633,0.51699,5000
5,2,0.90,0.512633,0.51699,5000
6,2,0.95,0.512633,0.51699,5000
7,2,1.00,0.512633,0.51699,5000
8,3,0.85,0.512633,0.51699,5000
9,3,0.90,0.512633,0.51699,5000


Best params -> min_df=1, max_df=0.85


d:\AmazonReviewRatingPredictions\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


--- Validation — Iteration 2 (Stopwords + DF tune) ---
              precision    recall  f1-score   support

           1       0.62      0.66      0.64     20871
           2       0.45      0.42      0.43     20793
           3       0.43      0.40      0.41     20674
           4       0.46      0.43      0.44     20842
           5       0.59      0.68      0.63     20820

    accuracy                           0.52    104000
   macro avg       0.51      0.52      0.51    104000
weighted avg       0.51      0.52      0.51    104000

--- Test — Iteration 2 (Stopwords + DF tune) ---
              precision    recall  f1-score   support

           1       0.61      0.66      0.63     25890
           2       0.45      0.41      0.43     25983
           3       0.43      0.39      0.41     26226
           4       0.46      0.44      0.45     25849
           5       0.60      0.68      0.63     26052

    accuracy                           0.52    130000
   macro avg       0.51    

### Iteration 2 Summary

- Best parameters from validation search: **min_df = 1**, **max_df = 0.85**.
- Added English stopword removal and tuned document-frequency thresholds.
- Outcome: performance **decreased** versus Iteration 1.
  - Validation accuracy is about **0.517** and weighted F1 is about **0.513**.
  - Test accuracy is about **0.516** and weighted F1 is about **0.512**.

This indicates stopword removal is not helping this dataset with the current setup; keeping sentiment-carrying function words may be important for these reviews.

## Iteration 3 — Stopword Removal Only

In this iteration, we test only English stopword removal while keeping the rest of the Iteration 1 setup unchanged (TF-IDF with unigrams + bigrams and `max_features=5000`).

- Removed: `min_df` tuning
- Removed: `max_df` tuning
- Kept: `stop_words='english'`

This isolates the effect of stopword removal by itself.

In [37]:
# Use TF-IDF with unigrams + bigrams (best from Iteration 1)
vectorizer_v3 = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf_v3 = vectorizer_v3.fit_transform(X_train)
X_val_tfidf_v3 = vectorizer_v3.transform(X_val)
X_test_tfidf_v3 = vectorizer_v3.transform(X_test)


In [38]:
# Tune regularization parameter C with cross-validation
C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
wef1_scores = []
val_acc = []
for C in C_values:
    lr = LogisticRegression(C=C, max_iter=2000, random_state=42)
    lr.fit(X_train_tfidf_v3, y_train)
    val_metrics = evaluate_model(lr, X_val_tfidf_v3, y_val, label=f"Validation — C={C}")
    wef1_scores.append(val_metrics["weighted avg"]["f1-score"])
    val_acc.append(val_metrics["accuracy"])
    print(f"C={C}: Validation Weighted F1={wef1_scores[-1]:.4f}, Accuracy={val_acc[-1]:.4f}")
# Select best C
best_C = C_values[np.argmax(wef1_scores)]
print(f"\nBest C: {best_C} with Validation Weighted F1: {max(wef1_scores):.4f}")

C=0.001: Validation Weighted F1=0.4662, Accuracy=0.4819
C=0.01: Validation Weighted F1=0.5134, Accuracy=0.5208
C=0.1: Validation Weighted F1=0.5456, Accuracy=0.5496
C=1.0: Validation Weighted F1=0.5503, Accuracy=0.5535
C=10.0: Validation Weighted F1=0.5488, Accuracy=0.5518
C=100.0: Validation Weighted F1=0.5494, Accuracy=0.5524

Best C: 1.0 with Validation Weighted F1: 0.5503


Default regularization value 1.0 is giving best results also removing stop words did not help and there was no increase in accuracy

## Iteration 4 - Using SVM

In this iteration, I tested an SVM-based classifier to see whether it can separate the review rating classes better than Logistic Regression.

### Changes I made
- I created a new TF-IDF feature set as `vectorizer_v4`.
- I increased `max_features` from **5000** to **10000**.
- I expanded the n-gram range from **(1, 2)** to **(1, 3)** so I could include trigrams.
- I added `min_df=2` to remove extremely rare terms.
- I transformed the training, validation, and test text into `X_train_tfidf_v4`, `X_val_tfidf_v4`, and `X_test_tfidf_v4`.
- I imported and trained `LinearSVC` with `class_weight="balanced"`, `max_iter=5000`, and `random_state=42`.
- I evaluated the SVM model on both validation and test data.


In [55]:
vectorizer_v4 = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 3),    # unigrams + bigrams + trigrams
    min_df=2,
)
X_train_tfidf_v4 = vectorizer_v4.fit_transform(X_train)
X_val_tfidf_v4 = vectorizer_v4.transform(X_val)
X_test_tfidf_v4 = vectorizer_v4.transform(X_test)

In [60]:
#Use svm
from sklearn.svm import LinearSVC
model_v3 = LinearSVC(class_weight="balanced", max_iter=10000, random_state=42)
model_v3.fit(X_train_tfidf_v4, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseud

In [61]:
#Evaluate SVM model
val_report_v3 = evaluate_model(model_v3, X_val_tfidf_v4, y_val, label="Validation — SVM")
test_report_v3 = evaluate_model(model_v3, X_test_tfidf_v4, y_test, label="Test — SVM")

Model name:LinearSVC, Validation — SVM Classification Report:
              precision    recall  f1-score   support

           1       0.62      0.71      0.66     20871
           2       0.49      0.42      0.45     20793
           3       0.49      0.44      0.47     20674
           4       0.51      0.44      0.47     20842
           5       0.62      0.75      0.68     20820

    accuracy                           0.56    104000
   macro avg       0.55      0.56      0.55    104000
weighted avg       0.55      0.56      0.55    104000

Model name:LinearSVC, Test — SVM Classification Report:
              precision    recall  f1-score   support

           1       0.62      0.72      0.66     25890
           2       0.49      0.42      0.45     25983
           3       0.49      0.44      0.46     26226
           4       0.50      0.44      0.47     25849
           5       0.62      0.75      0.68     26052

    accuracy                           0.55    130000
   macro avg 

## Final Comparison of Trained Models

I compared the trained models using the validation/test outcomes collected in earlier cells.

| Model / Iteration | Main Change | Validation Performance | Test Performance | My Takeaway |
|---|---|---|---|---|
| Base (LogReg + TF-IDF) | Unigrams, max_features=5000 | Accuracy: **0.5384**, Weighted F1: **0.5351** | Accuracy: **0.5356**, Weighted F1: **0.5320** | This was my baseline reference point. |
| Iteration 1 | Added bigrams (`ngram_range=(1,2)`) | Accuracy: **0.5535**, Weighted F1: **0.5503** | Accuracy: **0.5538**, Weighted F1: **0.5503** | This gave my best improvement over baseline. |
| Iteration 2 | Stopwords + `min_df/max_df` tuning | Accuracy: **~0.517**, Weighted F1: **~0.513** | Accuracy: **~0.516**, Weighted F1: **~0.512** | This reduced performance, so I did not keep it. |
| Iteration 3 | Stopword-only variant + C tuning check | No meaningful gain over Iteration 1 setup | No meaningful gain over Iteration 1 setup | I observed that removing stopwords was not beneficial here. |
| Iteration 4 (SVM) | LinearSVC + larger vocab + trigrams + balanced class weight | F1: 0.55 | No meaningful gain over Iteration 1 setup | I used this to test whether SVM handles class boundaries better than Logistic Regression. |

Overall, I found that **adding bigrams** was the most reliable improvement, while **stopword removal** did not help this dataset.

## 7. Using a LSTM model

In [52]:
# Use a smaller subset for faster LSTM experimentation
N_TRAIN_LSTM = 25000
N_EVAL_LSTM = 10000

# Reset index to keep text/label alignment simple for NN pipelines
X_train_lstm = X_train.iloc[:N_TRAIN_LSTM].reset_index(drop=True)
y_train_lstm = y_train.iloc[:N_TRAIN_LSTM].reset_index(drop=True)

X_val_lstm = X_val.iloc[:N_EVAL_LSTM].reset_index(drop=True)
y_val_lstm = y_val.iloc[:N_EVAL_LSTM].reset_index(drop=True)

X_test_lstm = X_test.iloc[:N_EVAL_LSTM].reset_index(drop=True)
y_test_lstm = y_test.iloc[:N_EVAL_LSTM].reset_index(drop=True)

print("LSTM subset sizes:")
print("X_train_lstm:", X_train_lstm.shape[0], "| y_train_lstm:", y_train_lstm.shape[0])
print("X_val_lstm  :", X_val_lstm.shape[0], "| y_val_lstm  :", y_val_lstm.shape[0])
print("X_test_lstm :", X_test_lstm.shape[0], "| y_test_lstm :", y_test_lstm.shape[0])

LSTM subset sizes:
X_train_lstm: 25000 | y_train_lstm: 25000
X_val_lstm  : 10000 | y_val_lstm  : 10000
X_test_lstm : 10000 | y_test_lstm : 10000


## 7.1 Tokenizing text data

### Notes on Tokenization

In this section, I converted the raw text reviews into numerical sequences so they can be fed into a neural network.

**Tokenization:**
I used Keras `Tokenizer` with a vocabulary size of `MAX_VOCAB_SIZE = 30,000`, meaning only the top 30,000 most frequent words in the training data are kept. Any word not in this vocabulary is replaced by the special `<OOV>` (Out-Of-Vocabulary) token. I fit the tokenizer **only on the training subset** to prevent data leakage from the validation or test sets.

**Sequence Padding:**
Since reviews have different lengths, I padded or truncated all sequences to a fixed length of `MAX_SEQUENCE_LENGTH = 100` tokens using **post-padding** and **post-truncation**. Post-padding appends zeros to the end of shorter sequences, which is preferred for sentiment analysis because the most meaningful words (at the beginning of a sentence) are preserved intact.

**Label Encoding:**
I converted the original rating labels (1–5) to zero-indexed labels (0–4) and stored them as NumPy `int32` arrays. This avoids pandas index alignment issues that can silently corrupt mini-batch training in Keras.

In [81]:
from keras_preprocessing.text import Tokenizer

# Tokenization settings for neural network input
MAX_VOCAB_SIZE = 30000
MAX_SEQUENCE_LENGTH = 150
OOV_TOKEN = "<OOV>"

# Fit tokenizer on LSTM training subset only to avoid data leakage
tokenizer_nn = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer_nn.fit_on_texts(X_train.tolist())

# Convert text to integer token sequences
X_train_seq = tokenizer_nn.texts_to_sequences(X_train.tolist())
X_val_seq = tokenizer_nn.texts_to_sequences(X_val.tolist())
X_test_seq = tokenizer_nn.texts_to_sequences(X_test.tolist())

In [82]:
#Apply padding to ensure uniform sequence lengths
from keras.utils import pad_sequences   # keras_preprocessing.sequence uses np.unicode_ removed in NumPy 2.0
X_train_padded = pad_sequences(X_train_seq, maxlen=MAX_SEQUENCE_LENGTH, padding="post", truncating="post")
# Post means padding/truncating is applied at the end of the sequence as beggining of sentence contains more important information for sentiment analysis.
X_val_padded = pad_sequences(X_val_seq, maxlen=MAX_SEQUENCE_LENGTH, padding = "post", truncating="post")
X_test_padded = pad_sequences(X_test_seq, maxlen=MAX_SEQUENCE_LENGTH, padding = "post", truncating="post")

In [90]:
# Convert labels [1-5] to [0-4] and store as NumPy arrays for Keras
# Using NumPy avoids pandas index alignment issues during mini-batch training.
y_train_nn = (y_train - 1).to_numpy(dtype=np.int32)
y_val_nn = (y_val - 1).to_numpy(dtype=np.int32)
y_test_nn = (y_test - 1).to_numpy(dtype=np.int32)

print("Label arrays ready:", y_train_nn.shape, y_val_nn.shape, y_test_nn.shape)
print("Label range:", y_train_nn.min(), "to", y_train_nn.max())

Label arrays ready: (416000,) (104000,) (130000,)
Label range: 0 to 4


In [84]:
print(X_train_padded[0])

[    9    10    16     5    21    43   194  1771   114    43  8139   589
     3   103  1771     4     3    23   253    88  1771     3    23   136
  1777   401     3    15     5  1771   489  2484   163   131   205     8
    48   539    48  1498 23462    48    38     1    40    11     2   331
     4   149    24  2179    15    16     5  1148     8  6217     7    15
    16    80     3    15  1771     9   624  4930  1442     4     3   237
   646    12    78    14     7   218    48  2505    13    87 15267    32
    78   386    20   717    78   773    15  7333   588    78    87  3562
    87    51   136    88  1995    13    87    15   830     4  2174    87
  9860    78     4 13859    78     4   131    78   166  5808     4    87
 15267     7    32    20   194  1771    14    31    87    51    16    88
  1771    78   773    41    23   142   155  2768    12     2  2642    80
    87    51     5   234    16     2]


## 7.2 Building LSTM Model

In [76]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [ ]:
# HYPERPARAMETERS v1
vocab_size = 30000        # top words from tokenizer
embedding_dim = 128      # size of word vectors (common: 100–300)
max_len = 200             # sequence length (based on your earlier choice)
lstm_units = 64          # number of LSTM neurons
dropout = 0.5

Using above hyperparams I observed that the model was overfitting on 25k Dataset. So next step would be increase dataset size and tune these params

In [85]:
# HYPERPARAMETERS v2
vocab_size = 30000        # top words from tokenizer
embedding_dim = 64       # size of word vectors (common: 100–300)
max_len = 100             # sequence length (based on your earlier choice)
lstm_units = 32          # number of LSTM neurons
dropout = 0.3

### Reducing Overfitting and Improving Accuracy — Hyperparameter Tuning

After observing heavy overfitting with the v1 hyperparameters, I scaled down the model complexity and introduced early stopping. Here is a summary of what I changed and why:

| Hyperparameter | v1 (Overfit) | v2 (Tuned) | Reason |
|---|---|---|---|
| `embedding_dim` | 128 | 64 | Smaller word vectors reduce model capacity, limiting memorization |
| `lstm_units` | 64 | 32 | Fewer LSTM neurons lower the risk of the model fitting noise |
| `max_len` | 200 | 100 | Most sentiment signals appear in the first 100 tokens; shorter sequences also reduce training time |
| `dropout` | 0.5 | 0.3 | With a smaller architecture, aggressive dropout was unnecessary and hurt convergence |

**Additional change — EarlyStopping callback:**
I added `EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)`. This halts training as soon as validation loss stops improving for 2 consecutive epochs and automatically restores the weights from the best epoch, preventing the model from continuing to overfit after the optimal point.

**Effect:**
Scaling down the architecture together with early stopping reduced the gap between training and validation accuracy, resulting in a more generalizable model.

**Results — Before vs After:**

| Metric | v1 (Overfit) | v2 (Tuned) |
|---|---|---|
| Training Accuracy | 80%+ | 61.32% |
| Validation Accuracy | ~44% | **56.95%** |
| Validation Loss | — | 1.0039 |
| Loss | — | 0.9206 |

The v1 model was heavily overfitting — training accuracy exceeded 80% while validation accuracy sat at only 44%, a gap of ~36 percentage points. After reducing model capacity and adding EarlyStopping, the validation accuracy improved to **56.95%** and the train/val gap narrowed significantly, indicating better generalization.

In [ ]:
model = Sequential()

# Embedding Layer
# mask_zero=True tells LSTM to ignore padded 0 tokens
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True))

# LSTM Layer
model.add(LSTM(lstm_units, dropout=dropout, recurrent_dropout=dropout))

# Dropout Layer
# Helps prevent overfitting by randomly dropping neurons
model.add(Dropout(dropout))

# Output Layer
# 5 classes → softmax for probability distribution
model.add(Dense(5, activation='softmax'))

model.compile(
    loss='sparse_categorical_crossentropy',  # for integer labels (1–5)
    optimizer='adam',                        # standard optimizer
    metrics=['accuracy']
)

d:\AmazonReviewRatingPredictions\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [87]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',        # watch validation loss
    patience=2,                # wait 2 epochs before stopping
    restore_best_weights=True  # revert to best model
)

In [91]:
history = model.fit(
    X_train_padded,
    y_train_nn,
    validation_data=(X_val_padded, y_val_nn),
    epochs=10,          # start small (can increase later)
    batch_size=64,      # balanced choice for speed + stability
    callbacks=[early_stop]
)

Epoch 1/10
6500/6500 ━━━━━━━━━━━━━━━━━━━━ 368s 57ms/step - accuracy: 0.4372 - loss: 1.2748 - val_accuracy: 0.5078 - val_loss: 1.1335
Epoch 2/10
6500/6500 ━━━━━━━━━━━━━━━━━━━━ 341s 52ms/step - accuracy: 0.5145 - loss: 1.1217 - val_accuracy: 0.5460 - val_loss: 1.0438
Epoch 3/10
6500/6500 ━━━━━━━━━━━━━━━━━━━━ 343s 53ms/step - accuracy: 0.5541 - loss: 1.0377 - val_accuracy: 0.5627 - val_loss: 1.0119
Epoch 4/10
6500/6500 ━━━━━━━━━━━━━━━━━━━━ 340s 52ms/step - accuracy: 0.5793 - loss: 0.9859 - val_accuracy: 0.5680 - val_loss: 0.9969
Epoch 5/10
6500/6500 ━━━━━━━━━━━━━━━━━━━━ 343s 53ms/step - accuracy: 0.5969 - loss: 0.9511 - val_accuracy: 0.5705 - val_loss: 0.9993
Epoch 6/10
6500/6500 ━━━━━━━━━━━━━━━━━━━━ 347s 53ms/step - accuracy: 0.6132 - loss: 0.9206 - val_accuracy: 0.5695 - val_loss: 1.0039


## 7.3 Bidirectional LSTM on Full Training Data

In [92]:
# Bidirectional LSTM model — Hyperparameters v2
from tensorflow.keras.layers import Bidirectional

bilstm_model = Sequential()

# Embedding layer — mask_zero ignores padding tokens
bilstm_model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, mask_zero=True))

# Bidirectional LSTM — reads sequence forward and backward
bilstm_model.add(Bidirectional(LSTM(lstm_units, dropout=dropout, recurrent_dropout=dropout)))

# Dropout for regularization
bilstm_model.add(Dropout(dropout))

# Output layer — 5 classes
bilstm_model.add(Dense(5, activation='softmax'))

bilstm_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

bilstm_model.summary()

Model: "sequential_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_11 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [93]:
early_stop_bilstm = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

bilstm_history = bilstm_model.fit(
    X_train_padded,
    y_train_nn,
    validation_data=(X_val_padded, y_val_nn),
    epochs=10,
    batch_size=64,
    callbacks=[early_stop_bilstm]
)

print("Best val_accuracy:", max(bilstm_history.history['val_accuracy']))

Epoch 1/10
6500/6500 ━━━━━━━━━━━━━━━━━━━━ 382s 58ms/step - accuracy: 0.5212 - loss: 1.1000 - val_accuracy: 0.5596 - val_loss: 1.0132
Epoch 2/10
6500/6500 ━━━━━━━━━━━━━━━━━━━━ 363s 56ms/step - accuracy: 0.5722 - loss: 0.9904 - val_accuracy: 0.5700 - val_loss: 0.9875
Epoch 3/10
6500/6500 ━━━━━━━━━━━━━━━━━━━━ 395s 61ms/step - accuracy: 0.5952 - loss: 0.9441 - val_accuracy: 0.5766 - val_loss: 0.9775
Epoch 4/10
6500/6500 ━━━━━━━━━━━━━━━━━━━━ 399s 61ms/step - accuracy: 0.6137 - loss: 0.9056 - val_accuracy: 0.5772 - val_loss: 0.9847
Epoch 5/10
6500/6500 ━━━━━━━━━━━━━━━━━━━━ 413s 63ms/step - accuracy: 0.6316 - loss: 0.8717 - val_accuracy: 0.5748 - val_loss: 1.0024
Best val_accuracy: 0.5771827101707458


### LSTM vs Bidirectional LSTM — Summary

I trained two sequence models using the same v2 hyperparameters and compared their performance:

The standard **LSTM** (trained on 25k samples) achieved a final training accuracy of **61.32%** and a validation accuracy of **56.95%** (val_loss: 1.0039).

The **Bidirectional LSTM** (trained on the full dataset) improved on both fronts — training accuracy reached **63.16%** and best validation accuracy hit **57.72%** (val_loss: 1.0024). The lower val_loss also indicates slightly better calibration.

The improvement from switching to a BiLSTM is modest but consistent. The Bidirectional LSTM reads each review both forward and backward, allowing it to capture context from both directions — for example, a negation word early in the sentence can still inform the meaning of words that appear later. This explains the small but meaningful gain in validation accuracy. The larger training set also contributed by giving the model more examples to learn generalizable patterns from.